# Semantic Segmentation with Ludwig

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ludwig-ai/ludwig/blob/main/examples/semantic_segmentation/semantic_segmentation.ipynb)

This notebook demonstrates semantic segmentation on the **CamSeq01** driving dataset using three different image decoders available in Ludwig:

| Decoder | Architecture | Encoder pairing | Best for |
|---------|-------------|-----------------|----------|
| `unet` | Symmetric encoder-decoder with skip connections | Built-in UNet encoder | General purpose, well-studied baseline |
| `segformer` | Lightweight MLP head on top of multi-scale features | `dinov2` / any ViT backbone | High accuracy, transformer features |
| `fpn` | Feature Pyramid Network top-down pathway | Any CNN (e.g. `efficientnet`) | Fast inference, good at multi-scale objects |

**GPU required** — an A100 or similar is recommended for the SegFormer run.

CamSeq01 contains 101 urban driving images at 960×720 with 32 semantic classes.

## Setup

In [ ]:
!pip install 'ludwig[vision]' --quiet

In [ ]:
import logging
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from ludwig.api import LudwigModel

## Dataset

Ludwig ships a built-in downloader for CamSeq01.  
The call below downloads and caches the dataset, then returns a `pd.DataFrame`
with two columns: `image_path` and `mask_path`.

In [ ]:
from ludwig.datasets import camseq

df = camseq.load(split=False)
print(f"Total samples: {len(df)}")
df.head()

In [ ]:
# Reserve first image for visual comparison at the end
pred_set = df[0:1].reset_index(drop=True)
train_set = df[1:]

print(f"Training samples : {len(train_set)}")
print(f"Prediction sample: {len(pred_set)}")

In [ ]:
# Quick look at the raw image and ground-truth mask
from PIL import Image as PILImage

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(PILImage.open(pred_set['image_path'][0]))
axes[0].set_title('Input image')
axes[0].axis('off')
axes[1].imshow(PILImage.open(pred_set['mask_path'][0]))
axes[1].set_title('Ground-truth mask (32 classes)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Option 1: UNet (default)

The classic UNet uses a symmetric encoder-decoder with skip connections.  
Ludwig's implementation supports a configurable number of stages (`num_stages`)
so you can trade off capacity against speed.

In [ ]:
unet_config = {
    'input_features': [
        {
            'name': 'image_path',
            'type': 'image',
            'preprocessing': {'num_processes': 4, 'height': 512, 'width': 512},
            'encoder': {'type': 'unet'},
        }
    ],
    'output_features': [
        {
            'name': 'mask_path',
            'type': 'image',
            'preprocessing': {
                'num_processes': 4,
                'height': 512,
                'width': 512,
                'num_classes': 32,
            },
            'decoder': {
                'type': 'unet',
                'num_stages': 4,  # configurable depth
                'num_fc_layers': 0,
                'conv_norm': 'batch',
            },
            'loss': {'type': 'softmax_cross_entropy'},
        }
    ],
    'combiner': {'type': 'concat', 'num_fc_layers': 0},
    'trainer': {
        'epochs': 50,
        'early_stop': 10,
        'batch_size': 4,
        'learning_rate': 0.0001,
    },
}

t0 = time.time()
unet_model = LudwigModel(unet_config, logging_level=logging.WARNING)
unet_stats, _, unet_output_dir = unet_model.train(
    dataset=train_set,
    experiment_name='seg_comparison',
    model_name='unet',
    skip_save_processed_input=True,
)
unet_time = time.time() - t0
print(f"UNet training time: {unet_time:.1f}s")

In [ ]:
unet_preds, _ = unet_model.predict(pred_set)
if not isinstance(unet_preds, pd.DataFrame):
    unet_preds = unet_preds.compute()
unet_pred_mask = torch.from_numpy(unet_preds['mask_path_predictions'].iloc[0])
print('UNet prediction mask shape:', unet_pred_mask.shape)

## Option 2: SegFormer (transformer backbone)

SegFormer pairs a DINOv2 vision-transformer backbone with a lightweight MLP
decoder head.  The hierarchical features from DINOv2 feed the SegFormer head
directly — no upsampling convolutions needed until the final prediction.

- Encoder: `dinov2` (`facebook/dinov2-base`, ~86M params pretrained)
- Decoder: `segformer` (hidden MLP projection → bilinear upsample)
- Best suited to: high-accuracy segmentation where GPU memory is not the bottleneck

In [ ]:
segformer_config = {
    'input_features': [
        {
            'name': 'image_path',
            'type': 'image',
            'preprocessing': {'num_processes': 4, 'height': 512, 'width': 512},
            'encoder': {
                'type': 'dinov2',
                'pretrained_model_name_or_path': 'facebook/dinov2-base',
                'use_pretrained': True,
                'trainable': True,
            },
        }
    ],
    'output_features': [
        {
            'name': 'mask_path',
            'type': 'image',
            'preprocessing': {
                'num_processes': 4,
                'height': 512,
                'width': 512,
                'num_classes': 32,
            },
            'decoder': {
                'type': 'segformer',
                'hidden_size': 256,
                'dropout': 0.1,
            },
            'loss': {'type': 'softmax_cross_entropy'},
        }
    ],
    'combiner': {'type': 'concat', 'num_fc_layers': 0},
    'trainer': {
        'epochs': 50,
        'early_stop': 10,
        'batch_size': 4,
        'learning_rate': 0.0001,
    },
}

t0 = time.time()
segformer_model = LudwigModel(segformer_config, logging_level=logging.WARNING)
segformer_stats, _, segformer_output_dir = segformer_model.train(
    dataset=train_set,
    experiment_name='seg_comparison',
    model_name='segformer',
    skip_save_processed_input=True,
)
segformer_time = time.time() - t0
print(f"SegFormer training time: {segformer_time:.1f}s")

In [ ]:
segformer_preds, _ = segformer_model.predict(pred_set)
if not isinstance(segformer_preds, pd.DataFrame):
    segformer_preds = segformer_preds.compute()
segformer_pred_mask = torch.from_numpy(segformer_preds['mask_path_predictions'].iloc[0])
print('SegFormer prediction mask shape:', segformer_pred_mask.shape)

## Option 3: FPN (lightweight)

The Feature Pyramid Network decoder builds a top-down pathway over multi-scale
feature maps produced by any CNN backbone.  Combined with EfficientNet it gives
a good accuracy/speed balance and fits comfortably in smaller GPU budgets.

- Encoder: `efficientnet` (pretrained on ImageNet, ~5M params)
- Decoder: `fpn` (lateral projections + top-down merging over 4 pyramid levels)
- Best suited to: production deployments where inference latency matters

In [ ]:
fpn_config = {
    'input_features': [
        {
            'name': 'image_path',
            'type': 'image',
            'preprocessing': {'num_processes': 4, 'height': 512, 'width': 512},
            'encoder': {
                'type': 'efficientnet',
                'use_pretrained': True,
                'trainable': True,
            },
        }
    ],
    'output_features': [
        {
            'name': 'mask_path',
            'type': 'image',
            'preprocessing': {
                'num_processes': 4,
                'height': 512,
                'width': 512,
                'num_classes': 32,
            },
            'decoder': {
                'type': 'fpn',
                'num_channels': 256,
                'num_levels': 4,
            },
            'loss': {'type': 'softmax_cross_entropy'},
        }
    ],
    'combiner': {'type': 'concat', 'num_fc_layers': 0},
    'trainer': {
        'epochs': 100,
        'early_stop': 10,
        'batch_size': 8,
        'learning_rate': 0.0001,
    },
}

t0 = time.time()
fpn_model = LudwigModel(fpn_config, logging_level=logging.WARNING)
fpn_stats, _, fpn_output_dir = fpn_model.train(
    dataset=train_set,
    experiment_name='seg_comparison',
    model_name='fpn',
    skip_save_processed_input=True,
)
fpn_time = time.time() - t0
print(f"FPN training time: {fpn_time:.1f}s")

In [ ]:
fpn_preds, _ = fpn_model.predict(pred_set)
if not isinstance(fpn_preds, pd.DataFrame):
    fpn_preds = fpn_preds.compute()
fpn_pred_mask = torch.from_numpy(fpn_preds['mask_path_predictions'].iloc[0])
print('FPN prediction mask shape:', fpn_pred_mask.shape)

## UNet depth ablation

Ludwig's UNet decoder exposes a `num_stages` parameter that controls how many
encoder/decoder stage pairs are stacked.  More stages → richer multi-scale
representations but also more parameters and longer training times.  The input
spatial dimensions must be divisible by `2^num_stages`.

Run the standalone sweep script to get the full table:

```bash
python unet_depth_sweep.py
```

Below we replicate a mini version for illustration.

In [ ]:
import yaml

SWEEP_BASE = {
    'input_features': [
        {
            'name': 'image_path',
            'type': 'image',
            'preprocessing': {'num_processes': 4, 'height': 512, 'width': 512},
            'encoder': {'type': 'unet'},
        }
    ],
    'output_features': [
        {
            'name': 'mask_path',
            'type': 'image',
            'preprocessing': {'num_processes': 4, 'height': 512, 'width': 512, 'num_classes': 32},
            'decoder': {'type': 'unet', 'num_fc_layers': 0, 'conv_norm': 'batch'},
            'loss': {'type': 'softmax_cross_entropy'},
        }
    ],
    'combiner': {'type': 'concat', 'num_fc_layers': 0},
    'trainer': {'epochs': 20, 'early_stop': 5, 'batch_size': 4, 'learning_rate': 0.0001},
}

sweep_results = []

for depth in [2, 3, 4, 5]:
    cfg = yaml.safe_load(yaml.dump(SWEEP_BASE))
    cfg['output_features'][0]['decoder']['num_stages'] = depth

    m = LudwigModel(cfg, logging_level=logging.WARNING)
    t0 = time.time()
    stats, _, _ = m.train(
        dataset=train_set,
        experiment_name='depth_sweep',
        model_name=f'unet_depth_{depth}',
        skip_save_processed_input=True,
    )
    elapsed = time.time() - t0

    n_params = sum(p.numel() for p in m.model.parameters() if p.requires_grad)

    val_loss = None
    try:
        val_loss = min(stats['validation']['combined']['loss'])
    except (KeyError, TypeError):
        pass

    sweep_results.append({
        'num_stages': depth,
        'trainable_params': f'{n_params:,}',
        'best_val_loss': round(val_loss, 4) if val_loss is not None else 'n/a',
        'training_time_s': round(elapsed, 1),
    })
    print(f"depth={depth}  params={n_params:,}  val_loss={val_loss}  time={elapsed:.1f}s")

sweep_df = pd.DataFrame(sweep_results)
sweep_df

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
depths = [r['num_stages'] for r in sweep_results]
times = [r['training_time_s'] for r in sweep_results]

ax1.bar(depths, times, color='steelblue', alpha=0.7, label='Training time (s)')
ax1.set_xlabel('UNet num_stages')
ax1.set_ylabel('Training time (s)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_xticks(depths)

val_losses = [r['best_val_loss'] for r in sweep_results if isinstance(r['best_val_loss'], float)]
if len(val_losses) == len(depths):
    ax2 = ax1.twinx()
    ax2.plot(depths, val_losses, 'o-', color='tomato', label='Best val loss')
    ax2.set_ylabel('Best val loss', color='tomato')
    ax2.tick_params(axis='y', labelcolor='tomato')

plt.title('UNet depth: training time vs validation loss')
plt.tight_layout()
plt.show()

## Visualisation

Compare the predicted segmentation maps from the three models side by side.

In [ ]:
def to_rgb(mask_tensor: torch.Tensor) -> np.ndarray:
    """Convert a CHW or HW class-index tensor to a displayable RGB image."""
    t = mask_tensor
    if t.dim() == 3:
        # (C, H, W) logits or class probabilities -> argmax
        t = t.argmax(dim=0)
    arr = t.cpu().numpy().astype(np.float32)
    arr = (arr - arr.min()) / max(arr.max() - arr.min(), 1e-5)
    return arr


input_img = PILImage.open(pred_set['image_path'][0])
gt_mask = PILImage.open(pred_set['mask_path'][0])

fig, axes = plt.subplots(1, 5, figsize=(22, 5))

axes[0].imshow(input_img)
axes[0].set_title('Input image')

axes[1].imshow(gt_mask)
axes[1].set_title('Ground truth')

axes[2].imshow(to_rgb(unet_pred_mask), cmap='tab20')
axes[2].set_title(f'UNet (depth=4)')

axes[3].imshow(to_rgb(segformer_pred_mask), cmap='tab20')
axes[3].set_title('SegFormer + DINOv2')

axes[4].imshow(to_rgb(fpn_pred_mask), cmap='tab20')
axes[4].set_title('FPN + EfficientNet')

for ax in axes:
    ax.axis('off')

plt.suptitle('Segmentation map comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Print a summary comparison table
comparison = pd.DataFrame([
    {'model': 'UNet (num_stages=4)', 'encoder': 'unet', 'decoder': 'unet', 'training_time_s': round(unet_time, 1)},
    {'model': 'SegFormer + DINOv2', 'encoder': 'dinov2', 'decoder': 'segformer', 'training_time_s': round(segformer_time, 1)},
    {'model': 'FPN + EfficientNet', 'encoder': 'efficientnet', 'decoder': 'fpn', 'training_time_s': round(fpn_time, 1)},
])
comparison